# All Split Techniques Comparison

Use this notebook to compare the latest available results across all split techniques: baseline, byclient, onedatasetout, and kfoldbyclient.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd()
if (cwd / 'baseline').exists() and (cwd / 'byclient').exists():
    experiments_root = cwd
elif (cwd / 'experiments').exists():
    experiments_root = cwd / 'experiments'
else:
    raise FileNotFoundError('Could not locate the experiments folder from the current working directory.')

split_techniques = ['baseline', 'byclient', 'onedatasetout', 'kfoldbyclient']
strategies = ['zscore', 'client_zscore', 'magnitude_features', 'magnitude_only', 'robust_clip']
ranking_columns = ['pr_auc', 'miss_rate', 'far', 'balanced_accuracy', 'f1']
ranking_ascending = [False, True, True, False, False]

runs = []
for split_technique in split_techniques:
    for strategy in strategies:
        results_root = experiments_root / split_technique / strategy / 'results'
        run_dirs = sorted([path for path in results_root.glob('run_*') if path.is_dir()]) if results_root.exists() else []
        latest_run = run_dirs[-1] if run_dirs else None
        runs.append({
            'split_technique': split_technique,
            'preprocessing_strategy': strategy,
            'latest_run': None if latest_run is None else latest_run.name,
            'run_path': None if latest_run is None else str(latest_run),
            'available': latest_run is not None,
        })

print(f'Using experiments root: {experiments_root}')
runs_df = pd.DataFrame(runs)
runs_df

## Load Latest Results


In [ ]:
global_frames = []
dataset_frames = []

for row in runs:
    if not row['available']:
        continue
    run_path = Path(row['run_path'])
    metrics_global = pd.read_csv(run_path / 'metrics_global.csv')
    metrics_by_dataset = pd.read_csv(run_path / 'metrics_by_dataset.csv')

    metrics_global['split_technique'] = row['split_technique']
    metrics_global['preprocessing_strategy'] = row['preprocessing_strategy']
    metrics_by_dataset['split_technique'] = row['split_technique']
    metrics_by_dataset['preprocessing_strategy'] = row['preprocessing_strategy']

    metrics_global['model_candidate'] = metrics_global['selected_candidate']
    metrics_by_dataset['model_candidate'] = metrics_by_dataset['selected_candidate']

    metrics_global['selected_candidate_label'] = (
        metrics_global['split_technique'] + ' | ' + metrics_global['preprocessing_strategy'] + ' + ' + metrics_global['model'] + ' (' + metrics_global['selected_candidate'] + ')'
    )
    metrics_by_dataset['selected_candidate_label'] = (
        metrics_by_dataset['split_technique'] + ' | ' + metrics_by_dataset['preprocessing_strategy'] + ' + ' + metrics_by_dataset['model'] + ' (' + metrics_by_dataset['selected_candidate'] + ')'
    )

    global_frames.append(metrics_global)
    dataset_frames.append(metrics_by_dataset)

all_global = pd.concat(global_frames, ignore_index=True) if global_frames else pd.DataFrame()
all_by_dataset = pd.concat(dataset_frames, ignore_index=True) if dataset_frames else pd.DataFrame()

print(f"Loaded {len(all_global)} global rows from {len(global_frames)} technique/strategy runs.")
print(f"Loaded {len(all_by_dataset)} dataset rows from {len(dataset_frames)} technique/strategy runs.")

## Overall Ranking


In [ ]:
display_columns = ['accuracy','balanced_accuracy','specificity','precision','recall','f1','roc_auc','pr_auc','far','miss_rate']
if all_global.empty:
    overall_comparison = pd.DataFrame(columns=['selected_candidate'])
else:
    overall_comparison = (
        all_global[['selected_candidate_label','split_technique','preprocessing_strategy','model','model_candidate','accuracy','balanced_accuracy','specificity','precision','recall','f1','roc_auc','pr_auc','far','miss_rate','tn','fp','fn','tp']]
        .rename(columns={'selected_candidate_label': 'selected_candidate'})
        .sort_values(ranking_columns, ascending=ranking_ascending)
        .reset_index(drop=True)
    )
display(overall_comparison if overall_comparison.empty else overall_comparison.assign(**{c: overall_comparison[c].round(4) for c in display_columns}))

## Best Per Technique


In [ ]:
if all_global.empty:
    best_per_technique = pd.DataFrame(columns=['split_technique', 'selected_candidate'])
else:
    best_per_technique = (
        all_global.sort_values(['split_technique', *ranking_columns], ascending=[True, *ranking_ascending])
        .groupby('split_technique', as_index=False)
        .first()
        [['split_technique','selected_candidate_label','preprocessing_strategy','model','selected_candidate','pr_auc','miss_rate','far','balanced_accuracy','f1']]
        .rename(columns={'selected_candidate_label': 'selected_candidate', 'selected_candidate': 'model_candidate'})
        .sort_values(ranking_columns, ascending=ranking_ascending)
        .reset_index(drop=True)
    )
display(best_per_technique if best_per_technique.empty else best_per_technique.assign(**{c: best_per_technique[c].round(4) for c in ['pr_auc','miss_rate','far','balanced_accuracy','f1']}))

## Best Per Dataset


In [ ]:
if all_by_dataset.empty:
    dataset_comparison = pd.DataFrame(columns=['dataset', 'selected_candidate'])
else:
    dataset_comparison = (
        all_by_dataset[['dataset','selected_candidate_label','split_technique','preprocessing_strategy','model','model_candidate','accuracy','balanced_accuracy','specificity','precision','recall','f1','roc_auc','pr_auc','far','miss_rate']]
        .rename(columns={'selected_candidate_label': 'selected_candidate'})
        .sort_values(['dataset', *ranking_columns], ascending=[True, *ranking_ascending])
        .reset_index(drop=True)
    )
display(dataset_comparison if dataset_comparison.empty else dataset_comparison.assign(**{c: dataset_comparison[c].round(4) for c in display_columns}))

## Interpretation


In [ ]:
if overall_comparison.empty:
    display(Markdown('No comparison results are available yet.'))
else:
    best_overall = overall_comparison.iloc[0]
    best_technique = best_per_technique.iloc[0] if not best_per_technique.empty else None
    strongest_generalization = best_per_technique.sort_values(['pr_auc', 'miss_rate'], ascending=[False, True]).iloc[0] if not best_per_technique.empty else None
    easiest_dataset = dataset_comparison.groupby('dataset', as_index=False).first().sort_values(['pr_auc', 'miss_rate'], ascending=[False, True]).iloc[0] if not dataset_comparison.empty else None
    hardest_dataset = dataset_comparison.groupby('dataset', as_index=False).first().sort_values(['pr_auc', 'miss_rate'], ascending=[True, False]).iloc[0] if not dataset_comparison.empty else None

    lines = [
        '### Key Takeaways',
        '',
        f"- **Best overall result:** `{best_overall['selected_candidate']}` ranks first with PR-AUC = **{best_overall['pr_auc']:.4f}**, miss rate = **{best_overall['miss_rate']:.4f}**, and FAR = **{best_overall['far']:.4f}**.",
        f"- **Best split technique under the current ranking:** `{strongest_generalization['split_technique']}` has the strongest top-ranked configuration in the current experiments." if strongest_generalization is not None else '',
        f"- **Best dataset-level performance:** `{easiest_dataset['dataset']}` appears easiest, while `{hardest_dataset['dataset']}` remains the most challenging across the evaluated techniques." if easiest_dataset is not None and hardest_dataset is not None else '',
        '- **Main pattern:** `magnitude_features` combined with nonlinear models remains the strongest setup across the different split definitions.'
    ]
    display(Markdown('\n'.join([line for line in lines if line])))